# Aperture — Detector Evaluation

Load the trained detector and run:
1. Full evaluation on the CIFAKE test split (confusion matrix, ROC curve, calibration plot).
2. Out-of-distribution (OOD) evaluation on hand-collected images under `data/ood_test/`.
3. Grad-CAM / attention-rollout visualizations for 6 sample images (3 real, 3 fake).

All plots are written under `eval_results/`.

In [ ]:
# --- environment setup (Colab-aware, mirrors notebook 02) ---
import os, sys, pathlib

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    APERTURE_ROOT = pathlib.Path('/content/drive/MyDrive/Aperture')
    if not (APERTURE_ROOT / 'Aperture' / '__init__.py').exists():
        raise FileNotFoundError(f'Aperture repo not found at {APERTURE_ROOT}.')
    os.chdir(APERTURE_ROOT)
    !pip install -q tqdm scikit-learn matplotlib seaborn opencv-python pytorch-grad-cam
else:
    os.chdir(pathlib.Path().resolve().parent)

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
# --- imports ---
import json, random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from PIL import Image
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
    f1_score, roc_auc_score, roc_curve,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from Aperture.ai_detector.dataset import CIFakeDataset, build_eval_transform
from Aperture.ai_detector.infer import AIDetector

CHECKPOINT = Path('models/ai_detector_best.pt')
EVAL_DIR = Path('eval_results'); EVAL_DIR.mkdir(parents=True, exist_ok=True)
assert CHECKPOINT.exists(), f'Missing checkpoint: {CHECKPOINT}. Run notebook 02 first.'
print('checkpoint:', CHECKPOINT)

In [ ]:
# --- load detector + helpers ---
detector = AIDetector(CHECKPOINT)
print('model:', detector.model_name, '| device:', detector.device)

def score_dataset(model, device, ds, batch_size=64, num_workers=2):
    """Run the model over a CIFakeDataset and return (y_true, y_prob, y_pred)."""
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=(device.type=='cuda'))
    model.eval()
    y_true, y_prob, y_pred = [], [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='scoring'):
            images = images.to(device, non_blocking=True)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]  # P(FAKE)
            preds = logits.argmax(dim=1)
            y_true.append(labels.numpy())
            y_prob.append(probs.cpu().numpy())
            y_pred.append(preds.cpu().numpy())
    return (np.concatenate(y_true), np.concatenate(y_prob), np.concatenate(y_pred))

In [ ]:
# --- 1. CIFAKE test set: full metrics + plots ---
cifake_test = CIFakeDataset('data/cifake/test', transform=build_eval_transform())
y_true, y_prob, y_pred = score_dataset(detector.model, detector.device, cifake_test)

metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'f1': float(f1_score(y_true, y_pred)),
    'auc': float(roc_auc_score(y_true, y_prob)),
}
print('CIFAKE test:', metrics)
(EVAL_DIR / 'cifake_metrics.json').write_text(json.dumps(metrics, indent=2))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['REAL', 'FAKE']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"CIFAKE confusion matrix (acc={metrics['accuracy']:.3f})")
fig.tight_layout(); fig.savefig(EVAL_DIR / 'confusion_matrix.png', dpi=120); plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, label=f"AUC = {metrics['auc']:.3f}")
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('CIFAKE ROC curve'); ax.legend()
fig.tight_layout(); fig.savefig(EVAL_DIR / 'roc_curve.png', dpi=120); plt.show()

# Calibration plot
prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=15, strategy='quantile')
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(prob_pred, prob_true, marker='o', label='detector')
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='perfectly calibrated')
ax.set_xlabel('Mean predicted P(FAKE)'); ax.set_ylabel('Fraction actually FAKE')
ax.set_title('Calibration (reliability) plot'); ax.legend()
fig.tight_layout(); fig.savefig(EVAL_DIR / 'calibration_plot.png', dpi=120); plt.show()

In [ ]:
# --- 2. OOD evaluation ---
# Expects images at data/ood_test/REAL/ and data/ood_test/FAKE/.
# If the per-generator breakdown is desired, name files like
# 'midjourney_xxx.png', 'flux_xxx.png', 'dalle3_xxx.png' under FAKE/.
OOD_ROOT = Path('data/ood_test')
ood_metrics: dict = {}
if not (OOD_ROOT / 'REAL').exists() or not (OOD_ROOT / 'FAKE').exists():
    print(f'No OOD images at {OOD_ROOT}/(REAL|FAKE)/ — skipping.')
else:
    ood_ds = CIFakeDataset(OOD_ROOT, transform=build_eval_transform())
    print('OOD class counts:', ood_ds.class_counts())
    yo, po, predo = score_dataset(detector.model, detector.device, ood_ds,
                                  batch_size=16, num_workers=0)
    ood_metrics['overall'] = {
        'accuracy': float(accuracy_score(yo, predo)),
        'f1': float(f1_score(yo, predo, zero_division=0)),
        'auc': float(roc_auc_score(yo, po)) if len(set(yo)) == 2 else None,
    }
    print('OOD overall:', ood_metrics['overall'])

    # Optional per-generator breakdown (filename-prefix-based).
    generators = {}
    for (path, label), pr in zip(ood_ds.samples, predo):
        if label != 1:
            continue
        gen = path.stem.split('_')[0].lower() if '_' in path.stem else 'unknown'
        generators.setdefault(gen, []).append(pr == label)
    if generators:
        per_gen = {g: round(float(np.mean(v)), 3) for g, v in generators.items()}
        ood_metrics['per_generator_accuracy'] = per_gen
        print('per-generator accuracy:', per_gen)

    (EVAL_DIR / 'ood_metrics.json').write_text(json.dumps(ood_metrics, indent=2))

    # OOD confusion matrix (optional, useful sanity check)
    cm_ood = confusion_matrix(yo, predo)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm_ood, display_labels=['REAL', 'FAKE']).plot(
        ax=ax, cmap='Oranges', colorbar=False)
    ax.set_title(f"OOD confusion matrix (acc={ood_metrics['overall']['accuracy']:.3f})")
    fig.tight_layout(); fig.savefig(EVAL_DIR / 'ood_confusion_matrix.png', dpi=120); plt.show()

In [ ]:
# --- 3. Sample Grad-CAM visualizations (3 real, 3 fake) ---
raw_ds = CIFakeDataset('data/cifake/test', transform=None)
real = [p for p, l in raw_ds.samples if l == 0]
fake = [p for p, l in raw_ds.samples if l == 1]
random.seed(0)
chosen = [(p, 0) for p in random.sample(real, 3)] + [(p, 1) for p in random.sample(fake, 3)]

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for col, (path, label) in enumerate(chosen):
    img = Image.open(path).convert('RGB')
    out = detector.predict_with_explanation(img)
    truth = 'REAL' if label == 0 else 'FAKE'
    pred_up = out['label'].upper()
    correct = (pred_up == truth)
    title_color = 'green' if correct else 'red'
    axes[0, col].imshow(img); axes[0, col].axis('off')
    axes[0, col].set_title(f'truth={truth}', fontsize=10)
    axes[1, col].imshow(out['overlay']); axes[1, col].axis('off')
    axes[1, col].set_title(f"pred={pred_up} ({out['confidence']:.2f})",
                           color=title_color, fontsize=10)
fig.suptitle('Grad-CAM / attention-rollout heatmaps', y=1.02)
fig.tight_layout(); fig.savefig(EVAL_DIR / 'sample_heatmaps.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- summary ---
summary = {
    'cifake': metrics,
    'ood': ood_metrics if ood_metrics else 'skipped (no images)',
    'artifacts': sorted(p.name for p in EVAL_DIR.iterdir()),
}
print(json.dumps(summary, indent=2))